# CSCI threshold ablation on MEVID-SC

**Question:** how strict should "person visible" be? This is the CSCI counterpart of the OSNet ablation in the Paper Extras notebook.

| Rule | Status |
|---|---|
| All frames | Already on Drive |
| Person found | Already on Drive |
| **≥50% of body visible (ours)** | Already on Drive |
| **≥25% visible** | **New run (cell 5)** |
| **≥75% visible** | **New run (cell 5)** |

**Run:** cells 1 to 5 on an A100 (an L4 also works). That means two CSCI runs, about 45 to 60 minutes each, with a progress bar. Pose checks are reused, nothing else is recomputed, and each run is saved to Drive when done.

**Then run cell 6** (standalone; it works after a restart). It gives one table: frames removed, IDF1, IDs per person, cross-camera recall, mAP and Rank-1 for all five rules, on the same 50 fair tests.

In [ ]:
# 1. CSCI threshold ablation on MEVID-SC: setup, frozen settings, and the
#    MEVID-SC test frames on local disk (about 10 minutes).
import os, glob, json, time, subprocess, sys, warnings, numpy as np, pandas as pd
warnings.filterwarnings('ignore')
from google.colab import drive
drive.mount('/content/drive')

SD = '/content/drive/Shareddrives/*'
def find(pats, required=True):
    for p in pats:
        h = sorted(glob.glob(p, recursive=True))
        if h:
            return h[0]
    if required:
        raise FileNotFoundError(pats[0])

SC_DIR = os.path.dirname(find([f'{SD}/Data/MEVID-SameClothes/SUBSET_REPORT.json',
                               f'{SD}/**/MEVID-SameClothes/SUBSET_REPORT.json']))
CS = os.path.dirname(find([f'{SD}/Data/MEVID-results/CSCI-SameClothes/'
                           'test_templates_csci_sc.npz',
                           f'{SD}/**/CSCI-SameClothes/test_templates_csci_sc.npz']))
WT = find([f'{SD}/Data/MEVID-results/CSCI/ez_eva02_vid_hybrid_extra_best.pth',
           f'{SD}/**/CSCI/ez_eva02_vid_hybrid_extra_best.pth'])
OUT = os.path.join(os.path.dirname(CS), 'OcclusionAware-SameClothes')
os.makedirs(OUT, exist_ok=True)

SETTINGS = {
    'check_every': 4,            # person check on every 4th frame; the
                                 # frames in between take the nearest check
    'pose_model': 'yolo11m-pose.pt', 'imgsz': 320, 'det_conf': 0.25,
    'keypoint_conf': 0.5,
    # a frame is KEPT if the person is found and at least this share of
    # the 17 body keypoints is visible (occlusion-aware); the second
    # variant only requires that the person is found
    'variants': {'v25': 0.25, 'v75': 0.75},   # new runs; 'found' and 'visible' (0.5) exist
    'min_kept_frames': 8,        # CSCI's clip length; below this the clip
                                 # keeps all its frames (unchanged)
    'precision_target': 0.90, 'n_splits': 25, 'seed': 2027}
with open(f'{OUT}/SETTINGS_ablation_frozen.json', 'w') as f:
    json.dump(SETTINGS, f, indent=2)

A = f'{SC_DIR}/annotations'
names = [l.strip() for l in open(f'{A}/test_name.txt') if l.strip()]
info = np.loadtxt(f'{A}/track_test_info.txt').astype(np.int64)
qidx = np.loadtxt(f'{A}/query_IDX.txt').astype(np.int64)
pid, cam = info[:, 2], info[:, 4]
LOCAL = '/content/sc_test'
if not os.path.isdir(f'{LOCAL}/bbox_test'):
    os.makedirs(LOCAL, exist_ok=True)
    import tarfile
    from tqdm.auto import tqdm
    t0 = time.time()
    with tarfile.open(find([f'{SC_DIR}/mevid-sc-bbox-test.tar'])) as tf_:
        for m in tqdm(tf_, total=len(names) + 600, desc='unpacking MEVID-SC test frames',
                      miniters=2000):
            tf_.extract(m, LOCAL)
    print(f'done in {time.time() - t0:.0f} s')
n_img = sum(len(f) for _, _, f in os.walk(f'{LOCAL}/bbox_test'))
print(f'{len(info)} clips, {len(np.unique(pid))} people, {len(names):,} '
      f'frames listed, {n_img:,} on disk | results -> {OUT}')
assert n_img >= len(names)


In [ ]:
# 2. Load the saved pose checks from the main MEVID-SC run (no recompute).
st = SETTINGS
parts = [np.load(f) for f in sorted(glob.glob(f'{OUT}/personcheck_[0-9]*.npz'))]
K = np.concatenate([p['k'] for p in parts])
FOUND = np.concatenate([p['found'] for p in parts])
VIS = np.concatenate([p['vis'] for p in parts])
print(f'{len(K):,} checked frames loaded')


In [ ]:
# 3. Build the filtered clips for the two new thresholds (same rule, only
#    the visibility threshold changes).
#    Every frame takes the result of the nearest checked frame.
o = np.argsort(K); K, FOUND, VIS = K[o], FOUND[o], VIS[o]
REPORT = []
for v, thr in st['variants'].items():
    new_names, new_info, changed, fallback = [], [], [], []
    for t, (s, e, p, oid, c) in enumerate(info):
        fr = np.arange(s, e + 1)
        lo, hi = np.searchsorted(K, s), np.searchsorted(K, e, 'right')
        kk = K[lo:hi]
        near = lo + np.abs(fr[:, None] - kk[None, :]).argmin(1) \
            if len(fr) * len(kk) < 4e6 else \
            lo + np.clip(np.searchsorted(kk, fr), 0, len(kk) - 1)
        keep = (FOUND[near] == 1) & (VIS[near] >= thr)
        if keep.sum() < st['min_kept_frames']:
            keep[:] = True; fallback.append(t)
        elif not keep.all():
            changed.append(t)
        start = len(new_names)
        new_names += [names[k] for k in fr[keep]]
        new_info.append([start, len(new_names) - 1, p, oid, c])
    root = f'/content/root_{v}'
    os.makedirs(root, exist_ok=True)
    with open(f'{root}/test_name.txt', 'w') as f:
        f.write('\n'.join(new_names) + '\n')
    np.savetxt(f'{root}/track_test_info.txt', np.array(new_info), fmt='%d')
    np.savetxt(f'{root}/query_IDX.txt', qidx, fmt='%d')
    for k, src in {'train_name.txt': f'{A}/train_name.txt',
                   'track_train_info.txt': f'{A}/track_train_info.txt',
                   'bbox_test': f'{LOCAL}/bbox_test'}.items():
        if not os.path.exists(f'{root}/{k}'):
            os.symlink(src, f'{root}/{k}')
    np.savez(f'{OUT}/filter_{v}.npz', changed=np.array(changed, int),
             fallback=np.array(fallback, int),
             kept=np.diff(np.array(new_info)[:, :2], axis=1)[:, 0] + 1)
    REPORT.append({'variant': v, 'keep rule': (f'person found, >= {thr:.0%} '
                   'of body visible' if thr else 'person found'),
                   'frames kept %': round(100 * len(new_names) / len(names), 1),
                   'clips changed': len(changed),
                   'clips unchanged (too few good frames)': len(fallback)})
display(pd.DataFrame(REPORT).set_index('variant'))


In [ ]:
# 4. CSCI: authors' code and released MEVID weights, unchanged, with the
#    same two patches as all earlier runs (feature dump; skip loading a
#    training clip).
REPO = '/content/ICCV-CSCI-Person-ReID'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/ppriyank/'
                    'ICCV-CSCI-Person-ReID.git', REPO], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'timm==1.0.9', 'einops', 'yacs', 'mmengine', 'decord',
                'tensorboardX', 'tensorboard', 'termcolor', 'tabulate',
                'h5py', 'scikit-learn', 'lmdb', 'albumentations',
                'torchinfo', 'grad-cam', 'opencv-python-headless'],
               check=True)
f = f'{REPO}/processor/eval_mevid.py'; src = open(f).read()
anchor = '    distmat = distmat.cpu()\n'
hook = (anchor +
        "    if local_rank == 0 and os.environ.get('CSCI_DUMP'):\n"
        "        import numpy as _np\n"
        "        _np.savez(os.environ['CSCI_DUMP'], "
        "qf=qf.float().cpu().numpy(), gf=gf.float().cpu().numpy(), "
        "q_pids=q_pids.numpy(), g_pids=g_pids.numpy(), "
        "q_camids=q_camids.numpy(), g_camids=g_camids.numpy())\n")
if 'CSCI_DUMP' not in src:
    assert src.count(anchor) == 1, 'eval_mevid.py changed upstream'
    src = src.replace(anchor, hook)
    if '\nimport os' not in src:
        src = 'import os\n' + src
    open(f, 'w').write(src)
f = f'{REPO}/data/dataset_loader.py'; src = open(f).read()
old_l = ('        self.return_index = return_index\n'
         '        self.__getitem__(0)\n')
new_l = ('        self.return_index = return_index\n'
         '        if not train:\n            self.__getitem__(0)\n')
if old_l in src:
    open(f, 'w').write(src.replace(old_l, new_l))
print('CSCI ready; weights:', WT)


In [ ]:
# 5. Run CSCI at the two new thresholds (about 45-60 min each on an A100,
#    longer on an L4). Live progress bar; saved to Drive when each finishes.
from tqdm.auto import tqdm
f_ = f'{REPO}/processor/eval_mevid.py'; src_ = open(f_).read()
for loop in ['    for n_iter, (imgs, pids, camids, clothes_id, clothes_ids, meta) in enumerate(dataloader):\n',
             '    for n_iter, (imgs, pids, camids, clothes_id, clothes_ids, meta, index) in enumerate(dataloader):\n']:
    tag = loop + "        if n_iter % 10 == 0 or n_iter == len(dataloader) - 1: print(f'CSCI_PROGRESS {n_iter + 1} {len(dataloader)}', flush=True)\n"
    if loop in src_ and 'CSCI_PROGRESS' not in src_.split(loop)[1][:200]:
        src_ = src_.replace(loop, tag)
open(f_, 'w').write(src_)
# 5. Run CSCI at the two new thresholds (about 45-60 min each; progress bar).
gidx = np.setdiff1d(np.arange(len(info)), qidx)
for v in st['variants']:
    OUTF = f'{OUT}/test_templates_csci_{v}.npz'
    if os.path.exists(OUTF):
        print('done already:', OUTF); continue
    DUMP = f'{OUT}/csci_dump_{v}.npz'
    if not os.path.exists(DUMP):
        env = dict(os.environ, CSCI_DUMP=DUMP, PYTHONPATH=REPO, PYTHONUNBUFFERED='1')
        cmd = ['torchrun', '--nproc_per_node=1', '--master_port=12399',
               'train_two_step.py', '--eval', '--resume',
               '--config_file', 'configs/mevid_eva02_l_cloth.yml',
               'DATA.ROOT', f'/content/root_{v}/', 'TEST.WEIGHT', WT,
               'SOLVER.SEED', '12345', 'TRAIN.TRAIN_VIDEO', 'True',
               'DATA.DATASET', 'mevid', 'MODEL.NAME',
               'ez_eva02_vid_hybrid_extra', 'TRAIN.COLOR_PROFILE', '17',
               'DATA.TEST_BATCH', '16', 'DATA.NUM_WORKERS', '2']
        t0 = time.time()
        print(f'Starting CSCI ({v}); loading data takes a few minutes before the bar appears ...')
        with open(f'{OUT}/csci_log_{v}.txt', 'w') as log:
            p = subprocess.Popen(cmd, cwd=REPO, env=env, text=True,
                                 stdout=subprocess.PIPE,
                                 stderr=subprocess.STDOUT)
            bar = None
            for line in p.stdout:
                log.write(line)
                if line.startswith('CSCI_PROGRESS'):
                    dn, tot = map(int, line.split()[1:3])
                    if bar is None or tot != bar.total or dn < bar.n:
                        if bar is not None:
                            bar.close()
                        bar = tqdm(total=tot, desc=f'{v}')
                    bar.update(dn - bar.n)
                elif any(k in line for k in ['Error', 'Traceback', 'out of memory']):
                    print(line.rstrip())
            p.wait()
            if bar is not None:
                bar.close()
        print(f'{v}: exit {p.returncode} after {(time.time() - t0) / 60:.0f} '
              'min (non-zero after the dump is CSCI\'s optional extra '
              'protocol; harmless)')
    assert os.path.exists(DUMP), f'CSCI stopped before dumping ({v}); see log'
    z = np.load(DUMP)
    assert (z['q_pids'] == pid[qidx]).all() and (z['g_pids'] == pid[gidx]).all()
    T = np.zeros((len(info), z['qf'].shape[1]), np.float32)
    T[qidx], T[gidx] = z['qf'], z['gf']
    T /= np.maximum(np.linalg.norm(T, axis=1, keepdims=True), 1e-8)
    np.savez(OUTF, templates=T, pid=pid, cam=cam)
    print('saved', OUTF)


In [ ]:
# 6. STANDALONE results: CSCI on MEVID-SC at every visibility threshold
#    (all frames | person found | >=25% | >=50% ours | >=75%). Same 50 fair
#    tests and calibrator as the main results. Works after a restart.
import os, glob, warnings, numpy as np, pandas as pd
warnings.filterwarnings('ignore')
from google.colab import drive
from tqdm.auto import tqdm
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from scipy.optimize import linear_sum_assignment
drive.mount('/content/drive')
RES = sorted(glob.glob('/content/drive/Shareddrives/*/Data/MEVID-results'))[0]
OA, CS = f'{RES}/OcclusionAware-SameClothes', f'{RES}/CSCI-SameClothes'
SCA = sorted(glob.glob('/content/drive/Shareddrives/*/Data/MEVID-SameClothes/annotations'))[0]
info = np.loadtxt(f'{SCA}/track_test_info.txt').astype(np.int64)
qidx = np.loadtxt(f'{SCA}/query_IDX.txt').astype(np.int64)
gidx = np.setdiff1d(np.arange(len(info)), qidx)
pid, cam = info[:, 2], info[:, 4]
L = info[:, 1] - info[:, 0] + 1
unit = lambda x: x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-8)
RULES = {'all frames': (f'{CS}/test_templates_csci_sc.npz', None),
         'person found': (f'{OA}/test_templates_csci_found.npz', f'{OA}/filter_found.npz'),
         'visible >= 25%': (f'{OA}/test_templates_csci_v25.npz', f'{OA}/filter_v25.npz'),
         'visible >= 50% (ours)': (f'{OA}/test_templates_csci_visible.npz', f'{OA}/filter_visible.npz'),
         'visible >= 75%': (f'{OA}/test_templates_csci_v75.npz', f'{OA}/filter_v75.npz')}

def accuracy(X):
    S = X[qidx] @ X[gidx].T; aps, r1 = [], []
    for i, q in enumerate(qidx):
        sp = pid[gidx] == pid[q]; sc = cam[gidx] == cam[q]
        good = np.flatnonzero(sp & ~sc); junk = np.flatnonzero(sp & sc)
        if not good.size:
            continue
        order = np.argsort(-S[i]); order = order[~np.isin(order, junk)]
        r = np.flatnonzero(np.isin(order, good))
        aps.append(np.mean((np.arange(len(r)) + 1) / (r + 1))); r1.append(float(r[0] == 0))
    return 100 * np.mean(aps), 100 * np.mean(r1)

def score(pred, p, c):
    _, pi = np.unique(p, return_inverse=True); _, ci = np.unique(pred, return_inverse=True)
    M = np.zeros((pi.max() + 1, ci.max() + 1)); np.add.at(M, (pi, ci), 1)
    r, cc = linear_sum_assignment(-M)
    _, ki = np.unique(c, return_inverse=True)
    Kc = np.zeros((M.shape[0], ki.max() + 1, M.shape[1])); np.add.at(Kc, (pi, ki, ci), 1)
    nq = M.sum(1)
    x_tot = (nq ** 2 - (Kc.sum(2) ** 2).sum(1)).sum()
    x_same = ((M ** 2).sum(1) - (Kc ** 2).sum((1, 2))).sum()
    return 100 * M[r, cc].sum() / len(p), (M > 0).sum(1).mean(), 100 * x_same / max(1, x_tot)

def pp(lab, p):
    _, pi = np.unique(p, return_inverse=True); _, ci = np.unique(lab, return_inverse=True)
    cell = np.bincount(pi * (ci.max() + 1) + ci).astype(float); nc = np.bincount(ci).astype(float)
    same = (nc * (nc - 1)).sum()
    return 100 * (cell * (cell - 1)).sum() / same if same else 100.0

CUTS = np.geomspace(0.001, 1.5, 120)
people = np.unique(pid); rng = np.random.default_rng(2027); SPL = []
for s in range(25):
    perm = rng.permutation(people); a = np.isin(pid, perm[:len(people) // 2])
    SPL.append([np.flatnonzero(a), np.flatnonzero(~a)])
ROWS = []
for rule, (fn, filt) in tqdm(RULES.items(), desc='thresholds'):
    if not os.path.exists(fn):
        print('not run yet:', rule); continue
    X = unit(np.load(fn)['templates'])
    removed = 0.0 if filt is None else 100 * (1 - np.load(filt)['kept'].sum() / L.sum())
    mAP, r1 = accuracy(X); res = []
    for G in tqdm(SPL, desc=f'   {rule}: 25 splits x 2 directions', leave=False):
        Ls = []
        for g in G:
            D = np.clip(1 - X[g] @ X[g].T, 0, None); D = (D + D.T) / 2; np.fill_diagonal(D, 0)
            Ls.append(linkage(squareform(D, checks=False), 'average'))
        for c, t in [(0, 1), (1, 0)]:
            best = CUTS[0]
            for cut in CUTS:
                if pp(fcluster(Ls[c], cut, 'distance'), pid[G[c]]) >= 90:
                    best = cut
            res.append(score(fcluster(Ls[t], best, 'distance'), pid[G[t]], cam[G[t]]))
    idf1, ids, xr = np.mean(res, 0)
    ROWS.append({'rule': rule, 'frames removed %': round(removed, 1), 'IDF1 %': round(idf1, 2),
                 'IDs per person': round(ids, 2), 'cross-camera recall %': round(xr, 2),
                 'mAP %': round(mAP, 2), 'Rank-1 %': round(r1, 2)})
ABL = pd.DataFrame(ROWS).set_index('rule')
ABL.to_csv(f'{OA}/ablation_threshold_csci_sc.csv')
display(ABL)
print('Saved to', f'{OA}/ablation_threshold_csci_sc.csv')
